<a href="https://colab.research.google.com/github/nat5natalia/hw1/blob/main/stemm_lemm_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Установка необходимых библиотек (если не установлены):
# !pip install datasets
# !pip install nltk
import nltk
import string
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Загрузка нужных данных NLTK:
nltk.download('punkt_tab') # Для токенизации
nltk.download('wordnet') # Для лемматизации
nltk.download('omw-1.4') # Часто нужно для корректной лемматизации (WordNet)
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Загружаем датасет ag_news (Hugging Face Datasets)
dataset = load_dataset("emotion")
lemmatizer = nltk.WordNetLemmatizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
def preprocess_with_stemming(text):
  # Приведение к нижнему регистру
  text = text.lower()
# Удаление знаков препинания
  text = text.translate(str.maketrans('', '', string.punctuation))
# Токенизация
  tokens = nltk.word_tokenize(text)
# Применение стемминга
  stemmer = nltk.PorterStemmer()
  stemmed_tokens = [stemmer.stem(token) for token in tokens]
  stemmed_tokens = [token for token in stemmed_tokens if token not in stop_words]
  return stemmed_tokens

In [ ]:
def preprocess_with_lemmatization(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if token not in string.punctuation
    ]
    lemmatized_tokens = [token for token in lemmatized_tokens if token not in stop_words]
    return lemmatized_tokens

In [ ]:
for i in range(5):
  first_example = dataset["train"][i]
  first_text = first_example["text"]
  print("Исходный текст:")
  print(first_text)
# Шаг 1: Токенизация без доп. очистки
  tokens = nltk.word_tokenize(first_text)
  print("\nТокены (без дополнительной очистки):")
  print(tokens)
# Сравним результаты стемминга и лемматизации
  stemmed_result = preprocess_with_stemming(first_text)
  lemmatized_result = preprocess_with_lemmatization(first_text)
  print("\nТокены после стемминга:")
  print(stemmed_result)
  print("\nТокены после лемматизации:")
  print(lemmatized_result)

Исходный текст:
i didnt feel humiliated

Токены (без дополнительной очистки):
['i', 'didnt', 'feel', 'humiliated']

Токены после стемминга:
['didnt', 'feel', 'humili']

Токены после лемматизации:
['didnt', 'feel', 'humiliate']
Исходный текст:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

Токены (без дополнительной очистки):
['i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake']

Токены после стемминга:
['go', 'feel', 'hopeless', 'damn', 'hope', 'around', 'someon', 'care', 'awak']

Токены после лемматизации:
['go', 'feel', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake']
Исходный текст:
im grabbing a minute to post i feel greedy wrong

Токены (без дополнительной очистки):
['im', 'grabbing', 'a', 'minute', 'to', 'post', 'i', 'feel', 'greedy', 'wrong']

Токены после стемминга:
['im', 'gra

In [ ]:
test_phrase = "Hello, world! How are you? easy-going person"
tokens = nltk.word_tokenize(test_phrase)
print("Токены без удаления пунктуации:")
print(tokens)

Токены без удаления пунктуации:
['Hello', ',', 'world', '!', 'How', 'are', 'you', '?', 'easy-going', 'person']


In [ ]:
text_no_punct = test_phrase.translate(str.maketrans('', '', string.punctuation))
tokens_no_punct = nltk.word_tokenize(text_no_punct)
print("Токены после удаления пунктуации до токенизации:")
print(tokens_no_punct)

Токены после удаления пунктуации до токенизации:
['Hello', 'world', 'How', 'are', 'you', 'easygoing', 'person']


In [ ]:
tokens_raw = nltk.word_tokenize(test_phrase)
tokens_filtered = [t for t in tokens_raw if t not in string.punctuation]
print("Токены после удаления пунктуации после токенизации:")
print(tokens_filtered)

Токены после удаления пунктуации после токенизации:
['Hello', 'world', 'How', 'are', 'you', 'easy-going', 'person']


In [ ]:
categories = [
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'comp.graphics',
    'comp.windows.x'
]

newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

In [ ]:
for i in range(5):
  first_example = newsgroups.data[i]
  first_text = first_example
  print("Исходный текст:")
  print(first_text)
# Шаг 1: Токенизация без доп. очистки
  tokens = nltk.word_tokenize(first_text)
  print("\nТокены (без дополнительной очистки):")
  print(tokens)
# Сравним результаты стемминга и лемматизации
  stemmed_result = preprocess_with_stemming(first_text)
  lemmatized_result = preprocess_with_lemmatization(first_text)
  print("\nТокены после стемминга:")
  print(stemmed_result)
  print("\nТокены после лемматизации:")
  print(lemmatized_result)

Исходный текст:
From: tpeng@umich.edu (Timothy Richard Peng)
Subject: Re: Duo 230 crashes aftersleep (looks like Apple bug!)
Organization: University of Michigan -- Ann Arbor
Lines: 7
Reply-To: tpeng@umich.edu
NNTP-Posting-Host: livy.ccs.itd.umich.edu
Originator: tpeng@livy.ccs.itd.umich.edu

if you have a memory card installed that's not one of apple's, this
may be the problem.  for a couple of months after the release of
the duo, some memory manufacturers were shipping duo memory cards w/
improper (non-self-refreshing) chips.  if you have a third party 
card, pull it and see if the sleep problem recurs.
  - tim  



Токены (без дополнительной очистки):
['From', ':', 'tpeng', '@', 'umich.edu', '(', 'Timothy', 'Richard', 'Peng', ')', 'Subject', ':', 'Re', ':', 'Duo', '230', 'crashes', 'aftersleep', '(', 'looks', 'like', 'Apple', 'bug', '!', ')', 'Organization', ':', 'University', 'of', 'Michigan', '--', 'Ann', 'Arbor', 'Lines', ':', '7', 'Reply-To', ':', 'tpeng', '@', 'umich.edu', 'NNT

In [ ]:
#1
raw_docs = [dataset['train'][i]['text'] for i in range(5)]
lemmatized_docs = [preprocess_with_lemmatization(doc) for doc in raw_docs]
docs_as_strings = [' '.join(tokens) for tokens in lemmatized_docs]
vectorizer1 = CountVectorizer(binary=True)
X1 = vectorizer1.fit_transform(docs_as_strings)
vocab1 = vectorizer1.get_feature_names_out()

In [ ]:
print(len(vocab1))
print(vocab1[:5])
print('10 первых элементов вектора 1 документа', X1[0].toarray()[0][:10])

24
['around' 'awake' 'care' 'damned' 'didnt']
10 первых элементов вектора 1 документа [0 0 0 0 1 0 1 0 0 0]


In [ ]:
#2
vectorizer2 = CountVectorizer(
    binary=True,
    tokenizer=preprocess_with_lemmatization,
    lowercase=False,
    token_pattern=None
)
X2 = vectorizer2.fit_transform(raw_docs)
vocab2 = vectorizer2.get_feature_names_out()

In [ ]:
print(len(vocab2))
print(vocab2[:5])
print('10 первых элементов вектора 1 документа', X2[0].toarray()[0][:10])

24
['around' 'awake' 'care' 'damned' 'didnt']
10 первых элементов вектора 1 документа [0 0 0 0 1 0 1 0 0 0]


In [ ]:
#3
vectorizer3 = CountVectorizer(binary=True)
X3 = vectorizer3.fit_transform(raw_docs)
vocab3 = vectorizer3.get_feature_names_out()

In [ ]:
print(len(vocab3))
print(vocab3[:5])
print('10 первых элементов вектора 1 документа', X3[0].toarray()[0][:10])

41
['about' 'am' 'and' 'around' 'awake']
10 первых элементов вектора 1 документа [0 0 0 0 0 0 0 0 0 1]


In [ ]:
print("Пример исходного текста 20 Newsgroups (первые 500 символов):")
print(newsgroups.data[0][:500])

Пример исходного текста 20 Newsgroups (первые 500 символов):
From: tpeng@umich.edu (Timothy Richard Peng)
Subject: Re: Duo 230 crashes aftersleep (looks like Apple bug!)
Organization: University of Michigan -- Ann Arbor
Lines: 7
Reply-To: tpeng@umich.edu
NNTP-Posting-Host: livy.ccs.itd.umich.edu
Originator: tpeng@livy.ccs.itd.umich.edu

if you have a memory card installed that's not one of apple's, this
may be the problem.  for a couple of months after the release of
the duo, some memory manufacturers were shipping duo memory cards w/
improper (non-self-r


In [ ]:
import re

def clean_text(text):
    # Удаляем строки заголовков (целиком), регистронезависимо
    header_pattern = r'^(From|Subject|Organization|Lines|Distribution|Reply-To|NNTP-Posting-Host|Originator|In article|Message-ID|Date|Newsgroups).*$'
    text = re.sub(header_pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)
    # Удаляем адреса электронной почты
    text = re.sub(r'\S+@\S+', '', text)
    # Удаляем строки, содержащие "writes:" или подобные
    text = re.sub(r'^.*writes:.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    # Удаляем HTML-теги и сущности
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-zA-Z]+;', '', text)
    # Оставляем только буквы, цифры, пробелы и базовые знаки препинания (.,!?-'")
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\-\'\"]', '', text)
    # Удаляем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
print("Примеры из Emotion (после очистки):")
for i in range(2):
    print(clean_text(dataset['train'][i]['text']))

print("\nПримеры из 20 Newsgroups (после очистки):")
for i in range(2):
    raw = newsgroups.data[i]
    cleaned = clean_text(raw)
    print(cleaned[:300] if len(cleaned) > 300 else cleaned)

Примеры из Emotion (после очистки):
i didnt feel humiliated
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

Примеры из 20 Newsgroups (после очистки):
if you have a memory card installed that's not one of apple's, this may be the problem. for a couple of months after the release of the duo, some memory manufacturers were shipping duo memory cards w improper non-self-refreshing chips. if you have a third party card, pull it and see if the sleep pro
Hi, there! I have a MAC LC and consider buying CD300. I've been told, however, that 1. The double speed of CD300 is achievable only on machines with SCSI-2. This is completely false. 2. The double speed is a prerequisite for PhotoCD multisession capability, which I need. This is also false. What you


In [ ]:
from nltk import pos_tag

def filter_nouns_adjs(tokens):
    # Удаляем не-буквенные токены
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    # Оставляем существительные (NN*) и прилагательные (JJ*)
    filtered = [word for word, tag in tagged if tag.startswith('N') or tag.startswith('J')]
    return filtered

def filter_nouns_adjs_verbs(tokens):
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    # Добавляем глаголы (VB*)
    filtered = [word for word, tag in tagged if tag.startswith('N') or tag.startswith('J') or tag.startswith('V')]
    return filtered

# Для Emotion
for i in range(2):
    text = dataset['train'][i]['text']
    # Сначала очистим текст
    clean = clean_text(text)
    tokens = nltk.word_tokenize(clean.lower())  # приводим к нижнему регистру
    print(f"\nEmotion текст {i+1} (очищенный): {clean[:100]}...")
    print("Только сущ и прил:", filter_nouns_adjs(tokens)[:10])
    print("Сущ, прил, глаголы:", filter_nouns_adjs_verbs(tokens)[:10])

# Для 20 Newsgroups
for i in range(2):
    text = newsgroups.data[i]
    clean = clean_text(text)
    tokens = nltk.word_tokenize(clean.lower())
    print(f"\n20 Newsgroups текст {i+1} (очищенный): {clean[:100]}...")
    print("Только сущ и прил:", filter_nouns_adjs(tokens)[:10])
    print("Сущ, прил, глаголы:", filter_nouns_adjs_verbs(tokens)[:10])


Emotion текст 1 (очищенный): i didnt feel humiliated...
Только сущ и прил: ['i', 'feel']
Сущ, прил, глаголы: ['i', 'didnt', 'feel', 'humiliated']

Emotion текст 2 (очищенный): i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and ...
Только сущ и прил: ['i', 'hopeless', 'damned', 'hopeful', 'someone', 'awake']
Сущ, прил, глаголы: ['i', 'go', 'feeling', 'hopeless', 'damned', 'hopeful', 'being', 'someone', 'cares', 'is']

20 Newsgroups текст 1 (очищенный): if you have a memory card installed that's not one of apple's, this may be the problem. for a couple...
Только сущ и прил: ['memory', 'card', 'apple', 'problem', 'couple', 'months', 'release', 'duo', 'memory', 'manufacturers']
Сущ, прил, глаголы: ['have', 'memory', 'card', 'installed', 'apple', 'be', 'problem', 'couple', 'months', 'release']

20 Newsgroups текст 2 (очищенный): Hi, there! I have a MAC LC and consider buying CD300. I've been told, however, that 1. The double sp...
Только сущ